# D03 — Holdout specifications (canonical, shipped)

This notebook displays the deterministic 30-perturbation holdout specifications used by P01 (CPA) and P02 (GEARS) to produce the manuscript's prediction h5ads. The canonical specifications are **shipped** with this repository under `precomputed/holdout_specs/holdout_specs.csv`; this notebook reads and summarises that file rather than regenerating it from scratch. As a result, the holdouts displayed here match exactly the `perturbation_target` field in the shipped severity-detail h5ads under `data/predictions/severity_details/`.

**Manuscript reference:** Methods §2.2 (Models and holdout design).

**Two split strategies, four conditions, 100 seeds each (400 holdouts total):**
- **random** — uniform draw of 30 perturbations from the 150-perturbation candidate pool, per seed.
- **stratified** — leverage-quintile stratified draw (6 perturbations from each of 5 leverage quintiles), per seed.

**150-perturbation candidate pool.** The pool is the set of model-eligible perturbations in each cell type's processed mid-HVG atlas — a deterministic construction from the training pipeline. The pool is not separately stored; it can be recovered as the union of all 200 random+stratified holdouts per cell type (verified in the final cell of this notebook).

**See also:** `TRAINING_PROVENANCE.md` for the author scripts that produced the canonical specs and the rationale for shipping them rather than regenerating.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import PRECOMPUTED_DIR

CANONICAL_SPECS = PRECOMPUTED_DIR / "holdout_specs" / "holdout_specs.csv"
print(f"Canonical specs: {CANONICAL_SPECS}")
print(f"Exists: {CANONICAL_SPECS.exists()}")

Canonical specs: precomputed/holdout_specs/holdout_specs.csv
Exists: True


## Load and summarise the canonical holdout specs

In [2]:
specs = pd.read_csv(CANONICAL_SPECS)
print(f"Total holdouts: {len(specs)}")
print(f"Columns: {specs.columns.tolist()}")
print()
print("Per-condition counts (expected: 100 seeds per (cell_type, split_type)):")
print(specs.groupby(["cell_type", "split_type"]).size().to_string())
print()
print("Seed range per condition:")
print(specs.groupby(["cell_type", "split_type"])["seed"].agg(["min", "max", "nunique"]).to_string())
print()
print("First two rows (truncated perturbation_list for display):")
display = specs.head(2).copy()
display["perturbation_list"] = display["perturbation_list"].str[:80] + "..."
print(display.to_string(index=False))

Total holdouts: 400
Columns: ['cell_type', 'split_type', 'seed', 'perturbation_list']

Per-condition counts (expected: 100 seeds per (cell_type, split_type)):
cell_type  split_type
K562       random        100
           stratified    100
RPE1       random        100
           stratified    100

Seed range per condition:
                      min  max  nunique
cell_type split_type                   
K562      random        1  100      100
          stratified    1  100      100
RPE1      random        1  100      100
          stratified    1  100      100

First two rows (truncated perturbation_list for display):
cell_type split_type  seed                                                                   perturbation_list
     K562     random     1 ANAPC5;C12orf45;CENPH;COQ5;COX6C;DAP3;DHX15;DYNLL1;EDC4;EXOC3;FNBP4;GON4L;HAUS8;...
     K562     random     2 AFG3L2;ANAPC13;BRF1;CCP110;CENPH;CFL1;CHAF1A;CPNE7;DDX19B;DDX52;FASTKD5;GTF2H2C;...


## Verify the 150-perturbation candidate pool

For each cell type, the union of perturbations across all 200 random+stratified holdouts should equal the canonical 150-perturbation training pool. This sanity check confirms that the pool is fully covered by the holdout draws and that the shipped specs are self-consistent.

In [3]:
for cell in ("K562", "RPE1"):
    g = specs[specs["cell_type"] == cell]
    perts = set()
    for plist in g["perturbation_list"]:
        perts.update(plist.split(";"))
    print(f"{cell}: {len(perts)} unique perturbations across {len(g)} holdouts "
          f"(expected: 150)")
    if len(perts) != 150:
        print(f"  WARNING: expected 150, found {len(perts)}")
print()
print("If both show exactly 150 unique perturbations, the canonical pool is fully covered.")

K562: 150 unique perturbations across 200 holdouts (expected: 150)
RPE1: 150 unique perturbations across 200 holdouts (expected: 150)

If both show exactly 150 unique perturbations, the canonical pool is fully covered.
